# FractalDSL — Notebook de Exemplos

Este notebook demonstra a **FractalDSL**, uma linguagem de domínio específico embutida em **Guile/Scheme** para descrever e renderizar fractais declarativamente.

## Pipeline de execução

```
arquivo .frac
     │
     ▼  fractal-reader.scm (parser de indentação + eval Scheme)
fractal alist  ──────────────────────────────────────────────────
     │                                                           │
     ▼  generate()                                              │
  modo coastline     →  deslocamento de ponto médio + IFS       │
  modo equation      →  grid de escape-time (Mandelbrot/Julia)  │
  modo IFS           →  jogo do caos (chaos game)               │
     │                                                           │
     ▼  export-csv()                                            │
  pontos.csv  (x, y, type, value)                               │
     │                                                           │
     ▼  render_fractal.py  ◄──────────────────────────────────── │
  imagem.png
```

A DSL cuida da **matemática e estrutura**. O Python cuida dos **pixels**.
Desde a versão 3-final, o comando `generate` em um arquivo `.frac` aciona automaticamente o renderer Python se houver um bloco `render`.

In [ ]:
import subprocess, os, sys
from pathlib import Path
from IPython.display import Image, display, Markdown

# Diretório base do projeto
BASE = Path(os.getcwd())
GUILE_DIR = BASE / 'guile'
FRAC_DIR  = BASE
PYTHON_RENDERER = BASE / 'python' / 'render_fractal.py'

def guile_available():
    try:
        r = subprocess.run(['guile', '--version'], capture_output=True, timeout=5)
        return r.returncode == 0
    except FileNotFoundError:
        return False

HAS_GUILE = guile_available()
print(f'Guile disponível: {HAS_GUILE}')
print(f'Base: {BASE}')

def run_frac(frac_file, timeout=120):
    """Executa um .frac via Guile. Retorna (stdout, stderr, returncode)."""
    if not HAS_GUILE:
        return '', 'Guile não instalado — usando CSV/PNG pré-gerados.', 1
    abs_frac = (FRAC_DIR / frac_file).resolve()
    result = subprocess.run(
        ['guile', '--no-auto-compile', '-c',
         f'(load "fractal-reader.scm") (run-frac-file "{abs_frac}")'],
        capture_output=True, text=True,
        cwd=str(GUILE_DIR), timeout=timeout
    )
    return result.stdout, result.stderr, result.returncode

def render_csv(csv_name, png_name, style='island', color=None, width=900, height=900):
    """Chama render_fractal.py diretamente sobre um CSV existente."""
    csv_path = GUILE_DIR / csv_name
    png_path = GUILE_DIR / png_name
    cmd = ['python3', str(PYTHON_RENDERER), str(csv_path), str(png_path),
           '--style', style, '--width', str(width), '--height', str(height)]
    if color:
        cmd += ['--color', color]
    subprocess.run(cmd, capture_output=True)
    return png_path

def show_frac(frac_file):
    """Mostra o conteúdo de um .frac file."""
    content = (FRAC_DIR / frac_file).read_text()
    display(Markdown(f'```\n{content}\n```'))

def show_png(png_name, width=600):
    png_path = GUILE_DIR / png_name
    if png_path.exists():
        display(Image(str(png_path), width=width))
    else:
        print(f'PNG não encontrado: {png_path}')

---
## Estrutura de dados: o fractal como *alist*

Internamente, todo fractal é uma **association list** imutável. Cada primitiva recebe um fractal e devolve um novo:

```scheme
; create-fractal retorna a estrutura base
(create-fractal "Mandelbrot")
; => ((name . "Mandelbrot") (equation . #nil) (ifs . #nil)
;     (coastline . #nil) (constants . ()) (iterations . 0)
;     (center . (0 0)) (zoom . 100) (resolution . (800 800)))

; Pipeline funcional — cada chamada devolve um novo fractal
(define mandelbrot
  (zoom
    (center
      (iterations
        (equation (create-fractal "Mandelbrot") "z=z^2+c")
        150)
      -0.5 0)
    100))
```

Não há mutação — `set-field` aplica `map` sobre a lista e devolve uma nova.

---
## Exemplo 1 — Ilha (`ilha.frac`)

**Modo:** coastline + decoração IFS  
**Técnica:** polígono de 7 vértices subdividido 6 vezes (midpoint displacement), decorado com Barnsley Fern em cada aresta.

In [ ]:
show_frac('ilha.frac')

In [ ]:
stdout, stderr, rc = run_frac('ilha.frac')
if rc == 0:
    print(stdout)
else:
    print(stderr or '(usando PNG pré-gerado)')
    render_csv('ilha.csv', 'ilha.png', style='island', color='mono')

show_png('ilha.png')

---
## Exemplo 2 — Floresta (`floresta.frac`)

**Modo:** coastline + decoração IFS  
**Diferença:** 9 pontos, `roughness 0.55` (costa mais recortada), `steps 120` (vegetação mais densa). Estilo `forest`: interior verde escuro, sem outline duro.

In [ ]:
show_frac('floresta.frac')

In [ ]:
stdout, stderr, rc = run_frac('floresta.frac')
if rc == 0:
    print(stdout)
else:
    print(stderr or '(usando PNG pré-gerado)')
    render_csv('floresta.csv', 'floresta.png', style='forest')

show_png('floresta.png')

---
## Exemplo 3 — Montanha (`montanha.frac`)

**Modo:** coastline + decoração IFS  
**Diferença:** `roughness 0.6` (mais irregular), estilo `mountain` → paleta monocromática, terra preenchida em cinza pedra, com outline.

In [ ]:
show_frac('montanha.frac')

In [ ]:
stdout, stderr, rc = run_frac('montanha.frac')
if rc == 0:
    print(stdout)
else:
    print(stderr or '(usando PNG pré-gerado)')
    render_csv('montanha.csv', 'montanha.png', style='mountain')

show_png('montanha.png')

---
## Exemplo 4 — England (`england.frac`)

**Modo:** coastline + decoração IFS  
**Diferença:** 12 pontos iniciais, `roughness 0.3` (menos aleatório), aproximando um contorno geográfico real.

In [ ]:
show_frac('england.frac')

In [ ]:
stdout, stderr, rc = run_frac('england.frac')
if rc == 0:
    print(stdout)
else:
    print(stderr or '(usando PNG pré-gerado)')
    render_csv('england.csv', 'england.png', style='island', color='teal')

show_png('england.png')

---
## Exemplo 5 — Mandelbrot (`mandelbrot.frac`)

**Modo:** escape-time  
**Técnica:** para cada pixel `(re, im)` do plano complexo, itera `z = z² + c` até `|z| > 2` ou atingir `iterations`. A cor do pixel é proporcional ao número de iterações. Pixels que nunca escapam (conjunto de Mandelbrot) ficam pretos.

In [ ]:
show_frac('mandelbrot.frac')

In [ ]:
stdout, stderr, rc = run_frac('mandelbrot.frac')
if rc == 0:
    print(stdout)
else:
    print(stderr or '(usando PNG pré-gerado)')
    render_csv('mandelbrot.csv', 'mandelbrot.png', style='island', color='green')

show_png('mandelbrot.png')

---
## Camada Scheme — primitivas diretamente

A camada `.frac` é apenas açúcar sintático sobre as primitivas Scheme. O mesmo fractal pode ser escrito diretamente em Scheme:

```scheme
; Equivalente a ilha.frac, mas em Scheme puro:
(define ilha
  (set-field
    (set-field (create-fractal "Ilha") 'iterations 10000)
    'coastline
    (list (cons 'points 7) (cons 'radius 1.0)
          (cons 'roughness 0.4) (cons 'depth 6)
          (cons 'decorate
            (list (cons 'steps 80) (cons 'scale 0.06)
                  (cons 'transforms (list ...)))))))

(export-csv ilha "ilha.csv")
```

E com a nova função `render-png!`, a DSL pode renderizar diretamente:

```scheme
(render-png! ilha "ilha.png")
; → exporta CSV, chama python3 render_fractal.py, gera PNG
```

In [ ]:
# Demonstração: executar Scheme diretamente (sem .frac) para gerar um Barnsley Fern
scheme_code = '''
(load "fractal-core.scm")
(load "fractal-ifs.scm")
(load "fractal-generate.scm")

(define barnsley
  (let* ((f (create-fractal "BarnsleyFern"))
         (f (set-field f 'iterations 15000))
         (f (set-field f 'ifs
              (list
                (list 'transform 0.01 (affine  0.00  0.00  0.00  0.16  0.00  0.00))
                (list 'transform 0.85 (affine  0.85  0.04 -0.04  0.85  0.00  1.60))
                (list 'transform 0.07 (affine  0.20 -0.26  0.23  0.22  0.00  1.60))
                (list 'transform 0.07 (affine -0.15  0.28  0.26  0.24  0.00  0.44))))))
    f))

(export-csv barnsley "barnsley.csv")
(display "Pontos gerados: ")
(display (length (generate barnsley)))
(newline)
'''

if HAS_GUILE:
    result = subprocess.run(
        ['guile', '--no-auto-compile', '-c', scheme_code],
        capture_output=True, text=True, cwd=str(GUILE_DIR), timeout=60
    )
    print(result.stdout or result.stderr)
    render_csv('barnsley.csv', 'barnsley.png', style='forest', color='limegreen', width=800, height=1000)
    show_png('barnsley.png', width=400)
else:
    print('Guile não disponível — exemplo teórico.')

---
## Renderização embutida na DSL

A partir da versão final, o arquivo `.frac` pode conter um bloco `render` que configura a saída. Quando `generate` é executado, a DSL chama automaticamente o renderer Python:

```
fractal Ilha
    ...

render               ← configura WIDTH/HEIGHT/COLOR/STYLE
    resolution 1200 1200
    color mono
    style island

generate Ilha        ← exporta ilha.csv E gera ilha.png automaticamente
```

No Scheme, o mesmo é feito com `render-png!`:

```scheme
; fractal-reader.scm exporta render-png!
(load "fractal-reader.scm")
(run-frac-file "../ilha.frac")   ; gera ilha.csv + ilha.png

; ou diretamente:
(render-png! meu-fractal "saida.png")
```

A função `render-png!` detecta automaticamente o caminho do renderer Python (local `../python/` ou Docker `/fractal/python/`) e lê as configurações de `render.cfg`.

---
## Parâmetros do renderer

| Parâmetro | Valores | Descrição |
|---|---|---|
| `--style` | `island`, `forest`, `mountain`, `cloud` | Estilo visual (só coastline) |
| `--color` | `green`, `ocean`, `fire`, `teal`, `limegreen`, `mono`, `gradient` | Paleta de cores |
| `--bg` | `#rrggbb` | Cor de fundo |
| `--width` / `--height` | inteiro | Dimensão do PNG em pixels |
| `--dpi` | inteiro | Resolução (padrão: 300) |
| `--alpha` | 0.0–1.0 | Transparência da nuvem de densidade |
| `--pt` | float | Tamanho do ponto no scatter |

### Modos de renderização por tipo de CSV

| `type` no CSV | Modo | Descrição |
|---|---|---|
| `coast` / `decor` | Coastline | Polígono + nuvem IFS |
| `escape` | Escape-time | Grid Mandelbrot/Julia |
| `point` | IFS legado | Heatmap de densidade |


---
## Conclusão

A FractalDSL demonstra como uma linguagem funcional com suporte a macros (Guile/Scheme) é uma plataforma natural para DSLs:

- **Homoiconicidade**: código e dados têm a mesma representação (listas), permitindo que `fractal-reader.scm` gere e `eval`ue código Scheme em tempo de execução.
- **Pipeline funcional imutável**: cada primitiva (`equation`, `iterations`, `zoom`, `ifs`, `with-depth`) é uma função pura que transforma um fractal em outro — composição sem efeitos colaterais.
- **Duas camadas**: a DSL `.frac` (externa, indentada) compila para Scheme puro, que por sua vez compila para uma alist que alimenta o engine de geração.
- **Renderização integrada**: `render-png!` fecha o loop — do `.frac` ao PNG em um único comando, sem sair do ambiente Scheme.